# Multi-Agent Debate — Bull, Bear, and a Judge

**Before this notebook:** finish `intro_llm_prompting.ipynb` first — this
builds directly on the `chat()` wrapper and the system-prompt idea from
there.

**The idea:** one LLM call gives you one opinion. But what if you deliberately
create *disagreement* — one agent arguing the bull case, one arguing the
bear case, both looking at the exact same facts — and then have a third
agent, the judge, weigh both arguments and decide?

This isn't about getting a "correct" trading signal. It's about a genuinely
useful pattern: **using an LLM's own inconsistency against itself**, forcing
it to argue both sides before committing to a verdict, rather than trusting
whatever it says on the first try.

In [2]:
import os, json
from dotenv import load_dotenv
load_dotenv()

PROVIDER = "anthropic"   # one of: "openai", "anthropic", "gemini", "mock"

CONFIG = {
    "openai":    {"api_key": os.environ.get("OPENAI_API_KEY"),    "base_url": None,
                  "model": "gpt-5-mini"},
    "anthropic": {"api_key": os.environ.get("ANTHROPIC_API_KEY"), "base_url": "https://api.anthropic.com/v1/",
                  "model": "claude-sonnet-5"},
    "gemini":    {"api_key": os.environ.get("GEMINI_API_KEY"),    "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
                  "model": "gemini-flash-latest"},   # alias -- always points at Google's current recommended flash model
}

def chat(messages, system=None, provider=PROVIDER, max_tokens=1000, verbose=False):
    """
    Send a list of {"role":..., "content":...} messages, get back the reply text.
    `system` is genuinely optional -- pass nothing, and no system message is sent at all.

    verbose=True prints WHY the response ended -- "stop" (finished naturally)
    vs "length" (cut off by max_tokens). Use this instead of guessing whether
    a short or odd-looking answer is a truncation problem or something else.
    """
    if provider == "mock":
        last_user = next((m['content'] for m in reversed(messages) if m['role']=='user'), '')
        sys_note = f' [as instructed: {system[:40]}...]' if system else ' [no system prompt given]'
        return f"[MOCK REPLY]{sys_note} You said: \"{last_user[:60]}\" -- imagine a real, helpful answer here."

    from openai import OpenAI
    cfg = CONFIG[provider]
    client = OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"]) if cfg["base_url"] else OpenAI(api_key=cfg["api_key"])

    full_messages = ([{"role": "system", "content": system}] if system else []) + messages
    resp = client.chat.completions.create(model=cfg["model"], messages=full_messages, max_tokens=max_tokens)

    choice = resp.choices[0]
    if verbose:
        print(f"[finish_reason: {choice.finish_reason}]")
        if choice.finish_reason == "length":
            print(f"[TRUNCATED -- hit the {max_tokens}-token limit before finishing. Raise max_tokens.]")
    return choice.message.content

print('chat() ready')

chat() ready


## 1. The setup — same facts, two agents, opposite instructions

Both agents see the **exact same headlines**. The only thing that differs is
their system prompt — one is instructed to build the strongest possible bull
case, the other the strongest possible bear case. Neither is "right" by
design; each is deliberately one-sided, on purpose.

In [3]:
STOCK = "COMI"
headlines = [
    "Bank posts record quarterly profit, beating analyst estimates",
    "Regulatory review announced into lending practices",
    "New CEO announces expansion into three new markets",
    "Analysts flag rising bad debt provisions",
]

BULL_SYSTEM = (
    "You are a bullish equity analyst. Given a list of headlines about a stock, "
    "build the STRONGEST possible case for buying it. Be persuasive but stay "
    "grounded in the actual headlines given. 3-4 sentences."
)
BEAR_SYSTEM = (
    "You are a skeptical, bearish equity analyst. Given a list of headlines about "
    "a stock, build the STRONGEST possible case for avoiding or selling it. Be "
    "persuasive but stay grounded in the actual headlines given. 3-4 sentences."
)

headline_text = "\n".join(f"- {h}" for h in headlines)
user_msg = [{"role": "user", "content": f"Headlines for {STOCK}:\n{headline_text}"}]

bull_case = chat(user_msg, system=BULL_SYSTEM)
bear_case = chat(user_msg, system=BEAR_SYSTEM)

print("BULL CASE:\n" + bull_case)
print()
print("BEAR CASE:\n" + bear_case)

BULL CASE:
COMI just delivered a record quarter, blowing past analyst expectations—clear evidence of operational strength and momentum heading into the next fiscal year. Under new leadership, the bank is moving aggressively to expand into three new markets, signaling a credible growth runway well beyond its current footprint. While a regulatory review and rising bad debt provisions warrant monitoring, these are the kind of routine risk-management steps a well-capitalized, disciplined bank takes proactively—especially one still posting record profits despite setting aside more for potential losses. In short, COMI is showing you can grow the top line, manage risk prudently, and expand strategically all at once—a combination that should reward patient shareholders as the new markets come online.

BEAR CASE:
Despite the headline-grabbing record profit, the details underneath are far more concerning: rising bad debt provisions suggest asset quality is deteriorating even as the bank books it

## 2. The judge — a third agent, seeing only the arguments

The judge never sees the raw headlines. It only sees the two competing
arguments, and has to weigh them — closer to how you'd actually want a
decision-maker to behave: not picking a side blindly, but being forced to
consider the strongest version of the opposing view before committing.

In [4]:
JUDGE_SYSTEM = (
    "You are a neutral investment committee judge. You will be given a bull case "
    "and a bear case for the same stock. Weigh both fairly, note which specific "
    "points you find more convincing and why, then give a final lean: BUY, SELL, "
    "or HOLD. Keep it to 5-6 sentences total."
)

judge_prompt = (
    f"BULL CASE:\n{bull_case}\n\n"
    f"BEAR CASE:\n{bear_case}\n\n"
    "Weigh these and give your verdict."
)

verdict = chat([{"role": "user", "content": judge_prompt}], system=JUDGE_SYSTEM)
print(verdict)

The bear case makes a sharper point: rising provisions alongside a regulatory review into lending practices is a combination worth taking seriously, since provisioning is forward-looking and regulators rarely open reviews without some cause. The bull case's framing of these as "routine risk management" glosses over the coincidence of timing—provisions rising just as lending practices come under scrutiny is not obviously benign. That said, a record profit despite higher provisions does show underlying earnings power and capital cushion, and new-market expansion could be a genuine growth catalyst rather than reckless overreach—leadership transitions often coincide with strategic resets. The key issue is that the bull case requires taking management's discipline on faith while two concrete red flags (credit quality, regulatory scrutiny) remain unresolved, and layering geographic expansion on top adds execution risk rather than mitigating it. Given the uncertainty is real and near-term (re

## 3. Cross-model debate — different providers, not just different prompts

Everything above used ONE provider for all three agents — the disagreement
came purely from different system prompts. A genuinely different experiment:
what if the bull, the bear, and the judge are each a **different model, from
a different company**? Does the debate feel different? Does one provider's
model consistently argue more aggressively than another's?

**To try this for real:** set up all three keys in your `.env`, then change
the `provider=` argument in each `chat()` call below to mix providers.

In [ ]:
# example structure for a real cross-provider run -- swap PROVIDER for each agent
bull_case_x  = chat(user_msg, system=BULL_SYSTEM, provider="openai")     # e.g. GPT
bear_case_x  = chat(user_msg, system=BEAR_SYSTEM, provider="anthropic")  # e.g. Claude
verdict_x    = chat(
    [{"role": "user", "content": f"BULL CASE:\n{bull_case_x}\n\nBEAR CASE:\n{bear_case_x}\n\nWeigh these and give your verdict."}],
    system=JUDGE_SYSTEM, provider="gemini"                               # e.g. Gemini as neutral judge
)
print("BULL (openai):\n", bull_case_x)
print("\nBEAR (anthropic):\n", bear_case_x)
print("\nJUDGE (gemini):\n", verdict_x)

**Worth reflecting on, honestly:** this pattern is genuinely useful — forcing
a model to construct opposing arguments before committing to an answer often
produces a more considered final verdict than asking it directly for an
opinion. But it is still just one LLM's judgment of two other LLMs' arguments,
built from a handful of headlines. Nothing here should be mistaken for a real
trading signal — it's a demonstration of an orchestration pattern, not a
strategy. If you wanted to actually wire this into the dashboard, this would
sit next to the sentiment panel as *additional context* for a human, exactly
like the news sentiment extra — never as something that places a trade on its
own.